# What your benchmark is not measuring

Two ways a reasonable-looking local benchmark quietly reports the wrong number,
both found the expensive way and both reproducible from this notebook alone â€”
no private data, no downloaded replays, no competitor's code.

1. **Replay-derived opponents are probably desynchronised by one turn.** If you
   rebuild an agent from `steps[t]["action"]`, every action fires a turn late.
   There is a one-line fix and an exact test for it.
2. **The default Kaggle notebook image is not the game you are competing in.**
   It ships an engine whose premium price curves are completely different from
   the server's â€” dumping produce looks safe there and floors your revenue on
   the ladder.

Neither is a strategy idea. Both are measurement bugs, and both silently
invalidate every experiment run on top of them, which is why they are worth more
than a strategy idea.

### First, pin the engine â€” this notebook is an example of why

The competition server runs `kaggle-environments` **1.32.5** (check
`module_version` in any downloaded replay). The default Kaggle notebook image
ships **1.29.3**. Finding 2 shows what that costs; this cell just makes sure
everything below runs on the version the ladder actually uses.

In [ ]:
import subprocess
import sys
from importlib.metadata import version

SERVER_VERSION = "1.32.5"
try:
    installed = version("kaggle-environments")
except Exception:  # pragma: no cover - metadata missing
    installed = "unknown"
if installed != SERVER_VERSION:
    print(f"image ships {installed}; installing {SERVER_VERSION} to match the server")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"kaggle-environments=={SERVER_VERSION}"],
        check=False,
    )

import json

from kaggle_environments import make
from kaggle_environments.envs.kaggriculture.kaggriculture import MARKET_PARAMS, market_price

# Behavioural check, not a version string: these are the numbers that decide the
# game, so verify them directly rather than trusting package metadata.
ON_SERVER_RULES = MARKET_PARAMS["STRAWBERRY"]["above_target"] == 1.6
print("engine loaded | strawberry above_target =",
      MARKET_PARAMS["STRAWBERRY"]["above_target"],
      "| matches server rules:", ON_SERVER_RULES)

## A small reference agent

The demonstrations need an agent that actually farms, but nothing here depends
on it being good â€” and it is not. It hires, gives each unit one tile to own and
walk to, plants strawberry when it can afford it and wheat otherwise, waters,
harvests, and sells everything. Entirely self-contained, so this notebook has no
dependency beyond `kaggle_environments` itself.

In [ ]:
def reference_agent(obs, config=None):
    """Wheat/strawberry loop: hire, claim a tile each, water, harvest, sell.

    Each unit owns one tile (assigned by index) and walks to it, so the farm
    fills instead of the units crowding one square. Nothing clever â€” it just has
    to produce enough volume that market saturation is visible.
    """
    seat = int(obs.get("player", 0) or 0)
    farm = obs["farms"][seat]
    private = obs["private"]
    tiles = farm["tiles"]
    seeds = dict(private.get("seeds", {}))
    shed = private.get("shed", {})
    money = farm["money"]
    day = int(obs.get("day", 0) or 0)

    # The starting quadrant is NW: x, y in 0..4 on the default 10x10 board.
    workable = [(x, y) for y in range(5) for x in range(5)]

    market = []
    if farm.get("hires_today", 0) < 8 and money > 150:
        market.append(["HIRE"])
    # Strawberry is an ongoing crop and pays far better; wheat is the cheap filler.
    if money > 1200 and seeds.get("STRAWBERRY", 0) < 2 and day < 18:
        market.append(["BUY_SEED", "STRAWBERRY", 2])
    elif money > 120 and seeds.get("WHEAT", 0) < 3 and day < 27:
        market.append(["BUY_SEED", "WHEAT", 3])
    for product, quantity in sorted(shed.items()):
        if quantity > 0:
            market.append(["SELL", product, int(quantity)])

    planned = {"WHEAT": 0, "STRAWBERRY": 0}

    def step_toward(px, py, tx, ty):
        if px < tx:
            return ["EAST"]
        if px > tx:
            return ["WEST"]
        if py < ty:
            return ["SOUTH"]
        if py > ty:
            return ["NORTH"]
        return ["PASS"]

    def unit_action(position, index):
        px, py = int(position[0]), int(position[1])
        here = tiles[py][px]
        if isinstance(here, dict) and here.get("kind") == "PLANT":
            if here.get("yield_units", 0) > 0:
                return ["HARVEST"]
            if not here.get("watered_today"):
                return ["WATER"]
        if isinstance(here, dict) and here.get("kind") == "WEED":
            return ["DIG"]
        if here is None:
            # PLANT is dropped entirely if more units request a crop than we
            # hold seed for, so each unit reserves against the same budget.
            for crop in ("STRAWBERRY", "WHEAT"):
                if seeds.get(crop, 0) - planned[crop] > 0:
                    planned[crop] += 1
                    return ["PLANT", crop]
        target = workable[index % len(workable)]
        if (px, py) == target:
            return ["PASS"]
        return step_toward(px, py, target[0], target[1])

    hands = [unit_action(position, index + 1)
             for index, position in enumerate(farm.get("hands") or [])]
    return {"farmer": unit_action(farm["farmer"], 0), "hands": hands, "market": market[:10]}


def run_match(player_a, player_b, seed, steps=720):
    env = make("kaggriculture", configuration={"episodeSteps": steps, "seed": seed}, debug=False)
    env.run([player_a, player_b])
    final = env.steps[-1]
    return env, [final[s]["observation"]["farms"][s]["money"] for s in (0, 1)]


env, money = run_match(reference_agent, "starter", 101)
print(f"reference agent {money[0]:,.0f}   starter {money[1]:,.0f}")

## Finding 1 â€” replay-derived opponents are off by one turn

Every ladder episode is downloadable, so a popular idea is to rebuild an
opponent as a "ghost" that replays their recorded actions. It is a genuinely
good idea: it gives you sparring partners from the live meta instead of the
public-notebook pool everyone has already saturated.

The trap is in the indexing. In a Kaggle replay, `steps[t]["action"]` is the
action an agent produced **from observation `t-1`** â€” it is stored alongside the
state it led to, not the state it was chosen from. So the action your ghost must
play at `obs["step"] == t` lives at `steps[t+1]["action"]`.

Get this wrong and nothing errors. The ghost just plays every move one turn
late: it walks to where the crop *was*, buys the seed a turn after it needed it,
and sells into a market that already moved. It still finishes the season, so
there is no failure to notice â€” only a silently weaker, subtly different
opponent.

You do not need anyone else's replay to test this. Generate one from self-play
and check whether your reconstruction reproduces it **exactly**.

In [ ]:
PASS = {"farmer": ["PASS"], "hands": [], "market": []}


def extract_tape(replay_steps, seat, shift):
    """Rebuild a seat's action sequence. shift=1 is correct, shift=0 is the bug."""
    tape = []
    for t in range(len(replay_steps)):
        source = t + shift
        if source < len(replay_steps) and seat < len(replay_steps[source]):
            tape.append(replay_steps[source][seat].get("action") or dict(PASS))
        else:
            tape.append(dict(PASS))
    return tape


def tape_agent(tape):
    def agent(obs, config=None):
        step = int(obs.get("step", 0) or 0)
        action = tape[step] if step < len(tape) else PASS
        return dict(action) if isinstance(action, dict) else dict(PASS)
    return agent


SEED = 4242
source_env, recorded = run_match(reference_agent, "starter", SEED)
replay = json.loads(json.dumps(source_env.toJSON()))["steps"]
print(f"recorded episode: {recorded[0]:,.0f} vs {recorded[1]:,.0f}")

for shift, label in ((0, "steps[t]  (the common bug)"), (1, "steps[t+1] (correct)")):
    tapes = [extract_tape(replay, s, shift) for s in (0, 1)]
    _, replayed = run_match(tape_agent(tapes[0]), tape_agent(tapes[1]), SEED)
    exact = all(abs(replayed[s] - recorded[s]) < 1e-9 for s in (0, 1))
    print(f"  {label}: {replayed[0]:>10,.0f} vs {replayed[1]:>10,.0f}   "
          f"{'EXACT' if exact else 'DOES NOT REPRODUCE'}")

Only the shifted reconstruction reproduces the episode. I validated the same
test on ten real ladder episodes: **10/10 byte-exact with the shift, 0/10
without.**

Make this assertion part of your harness. A ghost that cannot reproduce its own
source episode is not a model of that opponent, and every conclusion you draw
against it is measured against a strawman. In my case it had reversed the sign
of a real result â€” an opponent family I had rejected on this evidence turned out
to beat the agent I kept.

## Finding 2 â€” the default notebook image is not the game you are competing in

This one caught me while writing this notebook, which is the best possible
demonstration of it.

Replays record the version that produced them: every ladder episode I have says
**1.32.5**. The Kaggle notebook image ships **1.29.3**. I assumed that gap was
cosmetic. It is not â€” the *price curves themselves* are different, and they are
the single most important numbers in the game.

In [ ]:
# `above_target` and the resulting price at a full anchor-throughput glut, on the
# engine the notebook image ships versus the engine the ladder runs. The 1.29.3
# column is recorded here because the cell at the top of this notebook has
# already upgraded the session â€” run this notebook with that cell removed and the
# live column becomes the 1.29.3 one.
IMAGE_1_29_3 = {  # what a default Kaggle notebook gives you
    "STRAWBERRY": (0.40, 72), "MILK": (0.40, 96), "MELON": (0.90, 25), "WOOL": (0.80, 40),
}

print(f"{'resource':12s} | {'image 1.29.3':>22s} | {'server 1.32.5 (live)':>22s}")
print(f"{'':12s} | {'above_tgt':>10s} {'P(I0+T)':>11s} | {'above_tgt':>10s} {'P(I0+T)':>11s}")
for name, (old_target, old_price) in IMAGE_1_29_3.items():
    live_target = MARKET_PARAMS[name]["above_target"]
    live_price = market_price(name, MARKET_PARAMS[name]["I0"] + MARKET_PARAMS[name]["T"])
    print(f"{name:12s} | {old_target:10.2f} {old_price:11d} | {live_target:10.2f} {live_price:11d}")

print(f"\nthis session is running server rules: {ON_SERVER_RULES}")

On **1.29.3** the premium goods have `above_target` of 0.40â€“0.90, so a full
anchor-throughput glut still sells at $72 (strawberry) or $96 (milk). On
**1.32.5** the same resources are at 1.60â€“3.60 and that same volume sells at
**$1**.

That is not a tuning difference, it is a different economy. An agent developed
in a default notebook learns that dumping premium produce is roughly safe. On
the ladder it is catastrophic â€” and the local benchmark will never tell you,
because it is internally consistent and it always finishes the season.

There is a second, smaller divergence in the same gap: shed operations
(`PICKUP`, `DROP`, `PLACE`-to-shed) moved to resolve *before* the LOCKED-tile
guard. Three of the four tiles adjacent to the shed start locked, so on the older
build a unit standing there cannot reach the shed at all.

**Pin the version, and assert on behaviour rather than on the version string.**
The cell at the top of this notebook does both.

For reference, here is the full glut side of the price table on the engine the
ladder actually runs. `above_target` says how far the price moves when `T` units
past equilibrium are dumped, so anything above 1 collapses to the $1 floor at
ordinary volumes â€” and wheat, on a logarithmic branch, barely moves at all.

In [ ]:
print(f"{'resource':12s} {'base':>5s} {'above_func':>11s} {'above_target':>13s} {'P(I0+T)':>8s} {'P(I0+2T)':>9s}")
for name, params in MARKET_PARAMS.items():
    i0, throughput = params["I0"], params["T"]
    print(f"{name:12s} {params['base']:5d} {params['above_func']:>11s} "
          f"{params['above_target']:13.2f} "
          f"{market_price(name, i0 + throughput):8d} {market_price(name, i0 + 2 * throughput):9d}")

## Method notes that saved me the most time

**Measure the ceiling before you build the selector.** I nearly built an adaptive
policy that chose between behaviours based on the shop-unlock draw. Measuring
first: the perfect-oracle version was worth ~4k, and the observable feature I
planned to select on correlated with performance at **r = âˆ’0.197** â€” weak, and
the wrong sign. That selector would have been fitted noise, and it would have
looked adaptive.

**Use seeds as blocks and play both seats.** Seat asymmetry is real here. Score
each seed once, combining both seats, and treat that as your independent unit;
repeated deterministic games are not new evidence.

**A rejected change is rejected *conditional on the bottleneck that was binding
at the time*.** Three changes I had rejected with measurements turned out to be
false negatives once an upstream constraint lifted. When several consecutive
experiments fail with the *same* signature, stop varying the treatment and trace
the state â€” that found a day-3 insolvency that had invalidated five prior
conclusions in one run.

**When a change produces byte-identical output, your hypothesis is wrong, not
your implementation.** Three successive fixes to a stalled herd produced
identical results and cost a run each. One instrumented probe settled it: the
engine requires standing *on* the shed for `PICKUP`, while the agent accepted
adjacency, so 97 actions a game silently did nothing.

---

If you take one thing from this: before trusting any local result, check that
your engine matches the server's and that any replayed opponent can reproduce
its own source episode exactly. Both checks are a few lines. Neither is
interesting. Both of them invalidated weeks of my measurements.